# AgriVision Turmeric Inspector - MobileNetV2 Training
### Academic Mini-Project: Turmeric Leaf Disease Detection
**Target Classes (4):** `Leaf_Spot`, `Leaf_Blotch`, `Healthy`, `Not_turmeric_leaf`

This Google Colab / Jupyter notebook implements:
1. Automated class discovery from the `dataset/` directory.
2. Stratified 70% Train, 15% Validation, and 15% Test split (test set kept untouched).
3. Real-time data augmentation (flips, rotation, brightness).
4. Transfer Learning using pre-trained **MobileNetV2** (12 initial epochs with frozen base).
5. Fine-tuning top layers at a reduced learning rate.
6. Class weight computation to handle class imbalance.
7. Final test evaluation, classification report, and confusion matrix.
8. Export to `model/turmeric_model.keras`, `model/class_names.json`, and `model/turmeric_model.tflite`.

## 1. Setup Environment & Verify GPU

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Verify hardware acceleration
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow Version: {tf.__version__}")
print(f"Available GPUs: {len(gpus)}")
if gpus:
    print(f"Using GPU: {gpus[0].name}")
else:
    print("Running on CPU.")

## 2. Directory Paths & Hyperparameters

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
INITIAL_EPOCHS = 8
FINE_TUNE_EPOCHS = 4
LEARNING_RATE = 1e-3
FINE_TUNE_LR = 1e-5

BASE_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else "."
DATASET_PATHS = [
    "Turmeric Plant Disease Augmented Dataset",
    "Image Dataset for Turmeric Plant Leaf Disease Detection",
    "dataset_hf",
]

MODEL_DIR = os.path.join(BASE_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(MODEL_DIR, "turmeric_model.keras")
CLASS_NAMES_PATH = os.path.join(MODEL_DIR, "class_names.json")
TFLITE_SAVE_PATH = os.path.join(MODEL_DIR, "turmeric_model.tflite")
ROOT_TFLITE_PATH = os.path.join(BASE_DIR, "model.tflite")
CONFUSION_MATRIX_PATH = os.path.join(MODEL_DIR, "confusion_matrix.png")


## 3. Discover Dataset Classes and Collect Files
Subfolders in `dataset/` are automatically mapped to classes.

In [ ]:
import hashlib

TARGET_CLASSES = {
    "Aphids Disease", "Dry Leaf", "Healthy Leaf", "Leaf Blotch",
    "Leaf Spot", "Rhizome Disease Root", "Rhizome Healthy Root", "Not turmeric leaf"
}

def normalize_class_name(name):
    clean = name.replace("_", " ").strip()
    low = clean.lower()
    if "rhizome" in low and "disease" in low: return "Rhizome Disease Root"
    if "rhizome" in low and ("healthy" in low or "root" in low): return "Rhizome Healthy Root"
    if "dry" in low: return "Dry Leaf"
    if "aphid" in low: return "Aphids Disease"
    if "blotch" in low: return "Leaf Blotch"
    if "spot" in low: return "Leaf Spot"
    if "healthy" in low: return "Healthy Leaf"
    if "not turmeric" in low: return "Not turmeric leaf"
    return clean if clean in TARGET_CLASSES else None

def find_class_subdirs(root_dir):
    matched = []
    if not os.path.exists(root_dir): return matched
    root_norm = normalize_class_name(os.path.basename(root_dir))
    if root_norm:
        return [(root_norm, root_dir)]
    for current_root, dirs, _ in os.walk(root_dir):
        dirname = os.path.basename(current_root)
        norm = normalize_class_name(dirname)
        if norm and norm in TARGET_CLASSES and current_root != root_dir:
            matched.append((norm, current_root))
            dirs.clear()
    return matched

valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
seen_hashes = set()
by_class = {}

for ds_path in DATASET_PATHS:
    abs_path = os.path.abspath(ds_path)
    if not os.path.exists(abs_path): continue
    for norm_cls, class_dir in find_class_subdirs(abs_path):
        for root, _, files in os.walk(class_dir):
            for f in files:
                if os.path.splitext(f)[1].lower() in valid_exts:
                    fpath = os.path.join(root, f)
                    try:
                        with open(fpath, "rb") as fh:
                            h = hashlib.md5(fh.read()).hexdigest()
                        if h in seen_hashes: continue
                        seen_hashes.add(h)
                        by_class.setdefault(norm_cls, []).append(fpath)
                    except Exception:
                        pass

class_names = sorted(by_class.keys())
image_paths = []
labels = []
print(f"Found {len(class_names)} merged classes across datasets:")
for idx, cls in enumerate(class_names):
    imgs = by_class[cls]
    print(f"  [{idx}] {cls}: {len(imgs)} unique images")
    for p in imgs:
        image_paths.append(p)
        labels.append(idx)

image_paths = np.array(image_paths)
labels = np.array(labels)
print(f"\nTotal combined dataset samples: {len(image_paths)}")


## 4. Stratified Dataset Split: 70% Train / 15% Validation / 15% Test
The test set is kept completely untouched until final evaluation.

In [ ]:
# Split 1: 85% train+val, 15% test
paths_train_val, paths_test, labels_train_val, labels_test = train_test_split(
    image_paths, labels, test_size=0.15, stratify=labels, random_state=SEED
)

# Split 2: From 85%, train is 70% of total and val is 15% of total
val_fraction = 0.15 / 0.85
paths_train, paths_val, labels_train, labels_val = train_test_split(
    paths_train_val, labels_train_val, test_size=val_fraction, stratify=labels_train_val, random_state=SEED
)

print(f"Training set:   {len(paths_train)} ({len(paths_train)/len(image_paths)*100:.1f}%)")
print(f"Validation set: {len(paths_val)} ({len(paths_val)/len(image_paths)*100:.1f}%)")
print(f"Test set:       {len(paths_test)} ({len(paths_test)/len(image_paths)*100:.1f}%)")

## 5. Calculate Class Weights for Imbalance Handling

In [ ]:
unique_classes = np.unique(labels_train)
weights_arr = compute_class_weight(class_weight="balanced", classes=unique_classes, y=labels_train)
class_weights = {int(cls): float(w) for cls, w in zip(unique_classes, weights_arr)}
print("Computed class weights:")
for cls, w in class_weights.items():
    print(f"  {class_names[cls]}: {w:.3f}")


## 6. Build High-Performance `tf.data` Pipeline with Augmentation

In [ ]:
num_classes = len(class_names)

def load_and_preprocess(path, label):
    img_raw = tf.io.read_file(path)
    img = tf.io.decode_image(img_raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    one_hot = tf.one_hot(label, depth=num_classes)
    return img, one_hot

def create_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(paths_train, labels_train, shuffle=True)
val_ds = create_dataset(paths_val, labels_val, shuffle=False)
test_ds = create_dataset(paths_test, labels_test, shuffle=False)
print("Data pipelines ready.")


## 7. Model Architecture (MobileNetV2 Transfer Learning)

In [ ]:
# Data Augmentation block
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.15, seed=SEED),
    layers.RandomBrightness(0.2, seed=SEED)
], name="data_augmentation")

# MobileNetV2 Base (pre-trained on ImageNet)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze during phase 1

# Construct full network
inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="input_image")
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax", name="prediction")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="AgriVision_MobileNetV2")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 8. Phase 1 Training: Frozen Base (~12 Epochs)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
]

history_initial = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

## 9. Phase 2 Fine-Tuning: Unfreeze Top Layers at Low Learning Rate

In [ ]:
base_model.trainable = True
# Keep early feature layers frozen, fine-tune top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

total_epochs = INITIAL_EPOCHS + FINE_TUNE_EPOCHS
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=len(history_initial.history["loss"]),
    class_weight=class_weights,
    callbacks=callbacks
)

## 10. Evaluation on Untouched Test Set & Confusion Matrix

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\n>>> Test Accuracy: {test_acc * 100:.2f}%")

y_true = labels_test
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix Plot
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", xticklabels=class_names, yticklabels=class_names)
plt.title("AgriVision Turmeric Disease - Test Confusion Matrix", fontsize=13, fontweight="bold")
plt.ylabel("True Class", fontsize=11)
plt.xlabel("Predicted Class", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=150)
plt.show()
print(f"Saved confusion matrix plot to: {CONFUSION_MATRIX_PATH}")


## 11. Save Trained Model and Convert to TFLite

In [ ]:
# 1. Save Keras Model
model.save(MODEL_SAVE_PATH)
print(f"Saved Keras model to: {MODEL_SAVE_PATH}")

# 2. Save Class Names
with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(class_names, f, indent=2)
print(f"Saved class names to: {CLASS_NAMES_PATH}")

# 3. Export Quantized TFLite Model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(TFLITE_SAVE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"Saved TFLite model to: {TFLITE_SAVE_PATH} ({len(tflite_model) / (1024*1024):.2f} MB)")

import shutil
shutil.copyfile(TFLITE_SAVE_PATH, ROOT_TFLITE_PATH)
print(f"Copied TFLite model to root: {ROOT_TFLITE_PATH}")
